### Loading the dataset:

The dataset used here is Twitter Sentiment Analysis Dataset, which contains tweets labeled with their corresponding sentiments (positive, negative, neutral, irrelevant)

[Dataset Download link](https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis)

In [ ]:
import pandas as pd

df = pd.read_csv("sentimental_dataset/twitter_training.csv")
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [3]:
# Changing the column names to meaningful names
column_names = ["tweet_id", "entity", "sentiment", "content"]
df.columns = column_names

df.head()

,tweet_id,entity,sentiment,content
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [4]:
# Checking for the null values
df.isnull().sum()

tweet_id       0
entity         0
sentiment      0
content      686
dtype: int64

In [5]:
# Dropping all the rows containing the NULL Values.
df = df.dropna()
df.isnull().sum()

tweet_id     0
entity       0
sentiment    0
content      0
dtype: int64

### Text Preprocessing

In [6]:
# 1. Lowercasing

df["sentiment"] = df["sentiment"].str.lower()
df["content"] = df["content"].str.lower()

In [7]:
# 2. Remove URLs, mentions, hashtags, special chars

import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # remove URLs
    text = re.sub(r"@\w+", "", text)         # remove mentions
    text = re.sub(r"#\w+", "", text)         # remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # keep only letters
    return text

df["content"] = df["content"].apply(clean_text)

In [8]:
# 3. Tokenization

import nltk
from nltk.tokenize import word_tokenize

df["tokens"] = df["content"].apply(word_tokenize)

df["tokens"]

0        [i, am, coming, to, the, borders, and, i, will...
1        [im, getting, on, borderlands, and, i, will, k...
2        [im, coming, on, borderlands, and, i, will, mu...
3        [im, getting, on, borderlands, and, i, will, m...
4        [im, getting, into, borderlands, and, i, can, ...
                               ...                        
74676    [just, realized, that, the, windows, partition...
74677    [just, realized, that, my, mac, window, partit...
74678    [just, realized, the, windows, partition, of, ...
74679    [just, realized, between, the, windows, partit...
74680    [just, like, the, windows, partition, of, my, ...
Name: tokens, Length: 73995, dtype: object

In [9]:
# 4. Remove Stopwords

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

df["tokens"] = df["tokens"].apply(
    lambda x: [word for word in x if word not in stop_words]
)

In [10]:
df["tokens"]

0                                  [coming, borders, kill]
1                         [im, getting, borderlands, kill]
2                        [im, coming, borderlands, murder]
3                       [im, getting, borderlands, murder]
4                       [im, getting, borderlands, murder]
                               ...                        
74676    [realized, windows, partition, mac, like, year...
74677    [realized, mac, window, partition, years, behi...
74678    [realized, windows, partition, mac, years, beh...
74679    [realized, windows, partition, mac, like, year...
74680    [like, windows, partition, mac, like, years, b...
Name: tokens, Length: 73995, dtype: object

In [ ]:
# 5. Lemmatization

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sayan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sayan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sayan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sayan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [12]:
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    pos_tags = pos_tag(tokens)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]

In [13]:
df["tokens"] = df["tokens"].apply(lemmatize_tokens)

In [47]:
# Final Tokens after Text Preprocessing
df["tokens"]

0                                     [come, border, kill]
1                              [im, get, borderland, kill]
2                           [im, come, borderland, murder]
3                            [im, get, borderland, murder]
4                            [im, get, borderland, murder]
                               ...                        
74676    [realize, window, partition, mac, like, year, ...
74677    [realize, mac, window, partition, year, behind...
74678    [realize, window, partition, mac, year, behind...
74679    [realize, window, partition, mac, like, year, ...
74680    [like, window, partition, mac, like, year, beh...
Name: tokens, Length: 73995, dtype: object

In [ ]:
# Joining the Tokens:
df["clean_text"] = df["tokens"].apply(lambda x: " ".join(x))

In [15]:
df.head()

,tweet_id,entity,sentiment,content,tokens,clean_text
0,2401,Borderlands,positive,i am coming to the borders and i will kill you...,"[come, border, kill]",come border kill
1,2401,Borderlands,positive,im getting on borderlands and i will kill you all,"[im, get, borderland, kill]",im get borderland kill
2,2401,Borderlands,positive,im coming on borderlands and i will murder you...,"[im, come, borderland, murder]",im come borderland murder
3,2401,Borderlands,positive,im getting on borderlands and i will murder y...,"[im, get, borderland, murder]",im get borderland murder
4,2401,Borderlands,positive,im getting into borderlands and i can murder y...,"[im, get, borderland, murder]",im get borderland murder


## Model Building:

The models that I will be using are:

1. Logistic Regression
2. Naive Bayes
3. Decision Tree
4. Random Forest
5. XGBoost

In [48]:
# 1. Define Features (X) and Target (y)

from sklearn.preprocessing import LabelEncoder


X = df["clean_text"]      # processed text
y = df["sentiment"]       # labels

le = LabelEncoder()
y = le.fit_transform(y)

'''
The Label encoding is in the following way:
['irrelevant', 'negative', 'neutral', 'positive']
→ [0, 1, 2, 3]
'''

"\nThe Label encoding is in the following way:\n['irrelevant', 'negative', 'neutral', 'positive']\n→ [0, 1, 2, 3]\n"

In [17]:
# 2. Convert Text → Numerical (TF-IDF)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(X)

In [18]:
# 3. Split the data for training and testing

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=40
)

In [66]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model: str) -> dict:
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    return {
        "Accuracy" : accuracy, 
        "Precision" : precision,
        "Recall" : recall,
        "F1-score" : f1
    }

In [20]:
results = {}

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

log_reg_model = LogisticRegression(max_iter=1000, C=10)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    max_features="sqrt"
)

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5
)

log_reg_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

results["LogisticRegression"] = evaluate_model(log_reg_model)
results["RandomForest"] = evaluate_model(rf_model)
results["XGBoost"] = evaluate_model(xgb_model)


## Model Evaluation:

Evaluating the models using metrics such as accuracy, precision, recall, and F1-score.

In [68]:
evaluations = pd.DataFrame(results)
evaluations

,LogisticRegression,RandomForest,XGBoost
Accuracy,0.689713,0.398886,0.533218
Precision,0.689046,0.675662,0.580948
Recall,0.689713,0.398886,0.533218
F1-score,0.687344,0.308151,0.509005


## Testing the Models

In [ ]:
# Function to perform the Preprocessing:

def preprocess(text: str):
    text = text.lower()
    
    text = clean_text(text)
    tokens = word_tokenize(text)
    
    tokens = [token for token in tokens if token not in stop_words and token.isalpha()]
    
    tokens = lemmatize_tokens(tokens)
    
    return " ".join(tokens)

In [61]:
def test_model(original_text: str):
    # Step 1: Text Preprocessing
    text = preprocess(original_text)

    # Step 2: Vectorization
    text = vectorizer.transform([text])

    # Step 3: Prediction
    log_reg_model_pred = log_reg_model.predict(text)
    xgb_model_pred =xgb_model.predict(text)
    rf_model_pred =rf_model.predict(text)

    sentiments = ['irrelevant', 'negative', 'neutral', 'positive']

    # Displaying the final results
    print(f"Original Text is: {original_text}")
    print(f"Logistic Regression predicts: {sentiments[log_reg_model_pred[0]]}")
    print(f"Random Forest predicts: {sentiments[rf_model_pred[0]]}")
    print(f"XGBoost predicts: {sentiments[xgb_model_pred[0]]}")

In [65]:
print("Testing with a Negative statement:\n")
test_model("This is a Hate speech")

print(f"\n{'=' * 45}\n")

print("Testing with a Positive statement:\n")
test_model("I am a very good person")

Testing with a Negative statement:

Original Text is: This is a Hate speech
Logistic Regression predicts: negative
Random Forest predicts: negative
XGBoost predicts: negative


Testing with a Positive statement:

Original Text is: I am a very good person
Logistic Regression predicts: positive
Random Forest predicts: positive
XGBoost predicts: positive


## Saving the Model

In [43]:
import joblib

In [46]:
joblib.dump(log_reg_model, "models/LogisticRegressionModel.pkl")
joblib.dump(xgb_model, "models/XGBoostModel.pkl")
joblib.dump(rf_model, "models/RandomForestClassifierModel.pkl")

['models/RandomForestClassifierModel.pkl']

## Conclusion: Sentiment Analysis Model Evaluation

Based on the implementation and evaluation in this notebook, the sentiment analysis pipeline demonstrates a well-structured and effective approach to text classification.

### Preprocessing

The preprocessing pipeline, which includes lowercasing, text cleaning, tokenization, stopword removal, and lemmatization, successfully standardizes the input text and removes noise. This results in cleaner and more meaningful textual features, contributing positively to model performance.

### Vectorization

The use of TF-IDF vectorization provides a strong numerical representation of textual data. It effectively captures the importance of words across the dataset and works well for traditional machine learning models, especially in high-dimensional sparse text data.

### Model Performance

Among the models evaluated, Logistic Regression performs consistently well, offering a good balance between accuracy, efficiency, and interpretability. Other models like Random Forest and XGBoost also provide competitive results but come with higher computational complexity and tuning requirements.

### Trade-offs

* Simpler models like Logistic Regression are faster and easier to interpret but may not capture complex patterns.
* More advanced models can potentially improve performance but require more resources and tuning.
* TF-IDF is efficient but lacks semantic understanding compared to modern embedding techniques.

### Final Outcome

The overall pipeline is robust, efficient, and suitable for sentiment classification tasks. It achieves reliable performance without unnecessary complexity, making it a solid baseline solution for real-world applications.
